# v7 Python Authoring Quickstart

This notebook shows the new Python authoring layer on top of the existing `v7` training pipeline.

How it works:
- Python owns the project spec: model dimensions, template choice, data source, and train config.
- `TrainingProject.materialize()` calls the existing `version/v7/scripts/ck_run_v7.py init ... --generate-ir --generate-runtime` path.
- `TrainingProject.train()` calls the existing `version/v7/scripts/ck_run_v7.py train ...` path.
- `TrainingProject.prepare_viewers()` refreshes `ir_report.html`, prepares run-local viewer artifacts, and regenerates `ir_hub.html`.
- `dataset_viewer.html` and `attention.json` are conditional: they appear when the run has dataset manifests and tokenizer/probe artifacts.
- The actual runtime remains generated C code plus `libtrain.so`; Python is the authoring and orchestration layer.

How to start it from the repo root:
- `.venv/bin/jupyter lab notebooks/python_authoring/v7_training/02_v7_python_authoring_quickstart.ipynb`
- or `.venv/bin/jupyter notebook notebooks/python_authoring/v7_training/`

Open the notebook from inside the repo checkout so repo-root detection works.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


PosixPath('/home/antshiv/Workspace/C-Kernel-Engine/notebooks/python_authoring/v7_training')

In [2]:
from pathlib import Path
import importlib.util
import sys
from IPython.display import HTML, display

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ckernel_engine").exists() and (candidate / "version" / "v7").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not find the C-Kernel-Engine repo root from cwd={Path.cwd()}. "
        "Open this notebook from inside the repo checkout."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.find_spec("ckernel_engine")
if spec is None:
    raise RuntimeError(f"Repo root was found at {REPO_ROOT}, but Python still cannot import ckernel_engine")

import importlib
import ckernel_engine.v7.authoring as _authoring
import ckernel_engine.v7 as _v7
importlib.reload(_authoring)
importlib.reload(_v7)

from ckernel_engine.v7 import (
    DataSource,
    MaterializeOptions,
    TemplateSpec,
    TinyModelSpec,
    TokenizerPlan,
    TrainConfig,
    TrainingProject,
)

print(f"Imported ckernel_engine.v7 from repo root: {REPO_ROOT}")


Imported ckernel_engine.v7 from repo root: /home/antshiv/Workspace/C-Kernel-Engine


In [3]:
project = TrainingProject(
    run_name="python-ui-notebook-demo",
    model=TinyModelSpec(
        init="xavier_uniform",
        layers=2,
        vocab_size=256,
        embed_dim=128,
        hidden_dim=256,
        num_heads=8,
        num_kv_heads=4,
        context_len=128,
    ),
    template=TemplateSpec.builtin_template("qwen3"),
    tokenizer=TokenizerPlan(
        family="runtime_default",
        notes="Notebook keeps tokenizer ownership in the existing v7 runtime path.",
    ),
)

project


TrainingProject(run_name='python-ui-notebook-demo', model=TinyModelSpec(init='xavier_uniform', layers=2, vocab_size=256, embed_dim=128, hidden_dim=256, num_heads=8, num_kv_heads=4, context_len=128, rope_theta=1000000.0, kernel_policy='fp32_reference_first', adamw_beta1=0.9, adamw_beta2=0.999, adamw_eps=1e-08, adamw_weight_decay=0.01, seed=42), template=TemplateSpec(builtin='qwen3', document=None, source_path=None), tokenizer=TokenizerPlan(family='runtime_default', binary_dir=None, notes='Notebook keeps tokenizer ownership in the existing v7 runtime path.'), run_dir=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo'), repo_root=PosixPath('/home/antshiv/Workspace/C-Kernel-Engine'), python_exec='/home/antshiv/Workspace/C-Kernel-Engine/.venv/bin/python')

In [4]:
materialize_result = project.materialize(
    MaterializeOptions(
        generate_ir=True,
        generate_runtime=True,
        strict=True,
    )
)

materialize_result


Created tiny v7 training model at: /home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo
  entries=41  bump_bytes=2111744
  manifest=/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/weights_manifest.json
  bump=/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/weights.bump
Wrote IR1 train-forward: /home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir1_train_forward.json (ops=33 tensors=79)
Wrote report: /home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir1_train_report.json
Wrote IR2 train backward: /home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir2_train_backward.json (forward=33 backward=109 unresolved=0)
Wrote summary: /home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir2_train_summary.json
v7 train IR invariants: PASS (checks=5 failures=0 warnings=0)
Wrote training layout: /home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-

ExecutionResult(action='materialize', run_dir=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo'), command=('/home/antshiv/Workspace/C-Kernel-Engine/.venv/bin/python', '/home/antshiv/Workspace/C-Kernel-Engine/version/v7/scripts/ck_run_v7.py', 'init', '--run', '/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo', '--run-name', 'python-ui-notebook-demo', '--template', 'qwen3', '--train-mode', 'pretrain', '--train-bridge-lowering', 'legacy', '--train-checkpoint-policy', 'none', '--generate-ir', '--generate-runtime', '--strict', '--train-seed', '42', '--init', 'xavier_uniform', '--layers', '2', '--vocab-size', '256', '--embed-dim', '128', '--hidden-dim', '256', '--num-heads', '8', '--num-kv-heads', '4', '--context-len', '128', '--rope-theta', '1000000.0', '--kernel-policy', 'fp32_reference_first', '--adamw-beta1', '0.9', '--adamw-beta2', '0.999', '--adamw-eps', '1e-08', '--adamw-weight-decay', '0.01'), report_path=None, project_plan_path=

In [5]:
train_result = project.train(
    DataSource.inline_text(
        "C-Kernel-Engine notebook quickstart.",
        description="small inline training text",
    ),
    TrainConfig(
        backend="ck",
        strict=True,
        epochs=1,
        seq_len=8,
        total_tokens=64,
        grad_accum=2,
        lr=5e-4,
        parity_regimen="suggest",
        memory_check=False,
    ),
)

train_result


v7 CONTRACT REPORT
Layer | Handoff                           | Contract                                              | Status | Notes                         
------+-----------------------------------+-------------------------------------------------------+--------+-------------------------------
L1    | Spec -> IR/Kernel                 | v7 contract docs exist                                | PASS   | contracts-present             
L2    | Kernel sources -> v7 gate         | required fp32 backward-capable symbols                | PASS   | symbols-present=8             
L3    | Kernel API -> PyTorch parity      | v7 parity harness exists                              | PASS   | script=run_parity_1token_v7.py
L4    | Dev UX -> CI                      | v7 make targets wired                                 | PASS   | targets-present               
L5    | Kernel maps -> IR2 training lower | required training kernel IDs are registered and bound | PASS   | covered=13                    
S

[CK] Set 12 threads (auto=12)
[CK threadpool] Created 12 threads (1 main + 11 workers)


ExecutionResult(action='train', run_dir=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo'), command=('/home/antshiv/Workspace/C-Kernel-Engine/.venv/bin/python', '/home/antshiv/Workspace/C-Kernel-Engine/version/v7/scripts/ck_run_v7.py', 'train', '--run', '/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo', '--train-json-out', '/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/train_e2e_latest.json', '--train-mode', 'pretrain', '--backend', 'ck', '--train-epochs', '1', '--train-seq-len', '8', '--train-total-tokens', '64', '--train-grad-accum', '2', '--train-optimizer', 'adamw', '--train-lr', '0.0005', '--train-seed', '42', '--parity-regimen', 'suggest', '--train-bridge-lowering', 'legacy', '--train-checkpoint-policy', 'none', '--train-max-grad-norm', '0.0', '--train-strict', '--no-train-memory-check', '--prompt', 'C-Kernel-Engine notebook quickstart.'), report_path=PosixPath('/home/antshiv/.cache/ck-engine-v7/mode

In [6]:
viewer_artifacts = project.prepare_viewers()
viewer_artifacts


Generating report for: python-ui-notebook-demo
  - Missing optional files: ['memory_verification', 'training_parity_regimen', 'training_parity_xray', 'training_canary_summary', 'run_ledger', 'dataset_qc', 'dataset_profile', 'tokenizer_roundtrip', 'post_train_eval', 'stage_eval_matrix', 'training_epoch_sweep', 'run_index', 'sanity_overfit', 'parity_report', 'profile_latest', 'contract_report', 'parity_1token', 'qk_norm_backward_parity', 'inference_llama_parity', 'inference_llama_parity_prefill', 'inference_llama_parity_decode', 'inference_llama_autopsy', 'inference_llama_dump_index', 'fd_gradients', 'train_parity_epochs_3', 'train_parity_epochs_5', 'train_runtime_parity_realistic', 'train_runtime_parity_stress', 'replay_determinism', 'backprop_stitch_runtime', 'profile_summary', 'perf_stat_summary', 'flamegraph_manifest', 'cachegrind_summary', 'asan_summary', 'vtune_summary', 'advisor_summary', 'memory_signoff', 'perf_gate_report', 'embedding_dump', 'analysis_checkpoints', 'weight_healt

ViewerArtifacts(run_dir=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo'), models_root=PosixPath('/home/antshiv/.cache/ck-engine-v7/models'), ir_report=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir_report.html'), dataset_viewer=None, embeddings=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/embeddings.json'), attention=None, ir_hub=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/ir_hub.html'), hub_index=PosixPath('/home/antshiv/.cache/ck-engine-v7/models/runs_hub_index.json'))

## What To Inspect

After the cells above run, the important artifacts are:
- `python_authoring_plan.json`: Python-side project spec and command history
- `weights_manifest.json`: manifest-first training state and embedded template contract
- `ir1_train_forward.json`: train forward IR
- `ir2_train_backward.json`: backward IR
- `layout_train.json`: planned memory layout
- `train_e2e_latest.json`: latest train report exported by the existing `v7` runner
- `ir_report.html`: generated IR visualizer for this run
- `embeddings.json`: exported token embedding matrix when viewer prep runs against training weights
- `attention.json`: attention export when tokenizer/probe artifacts are available
- `dataset_viewer.html`: run-local dataset/token artifact viewer when dataset manifests are available
- `ir_hub.html`: shared run hub under the parent models root
- `generated_train_runtime_v7.c`: generated training runtime source
- `libtrain.so`: compiled generated runtime


In [7]:
run_dir = train_result.run_dir
display(HTML(project.notebook_artifact_dashboard_html(title="Quickstart Artifact Dashboard")))
sorted(p.name for p in run_dir.iterdir())


Artifact,Status,Path,Notes
Project Plan,ready,python_authoring_plan.json/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/python_authoring_plan.json,Python-side project spec and command history.
Weights Manifest,ready,weights_manifest.json/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/weights_manifest.json,Manifest-first training state and template contract.
Train Report,ready,train_e2e_latest.json/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/train_e2e_latest.json,"Latest train, sanity, or parity JSON report from the existing v7 runner."
IR Forward,ready,ir1_train_forward.json/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir1_train_forward.json,Train forward IR lowered by the existing v7 scripts.
IR Backward,ready,ir2_train_backward.json/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir2_train_backward.json,Backward IR and stitched train graph for generated runtime codegen.
Layout,ready,layout_train.json/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/layout_train.json,Planned memory layout for the training runtime.
Generated Train Runtime,ready,generated_train_runtime_v7.c/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/generated_train_runtime_v7.c,Generated C training runtime emitted by codegen.
Compiled Train Runtime,ready,libtrain.so/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/libtrain.so,Compiled shared library for the generated training runtime.
IR Report,ready,ir_report.html/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/ir_report.html,Generated IR visualizer for this run.
Embeddings Export,ready,embeddings.json/home/antshiv/.cache/ck-engine-v7/models/train/python-ui-notebook-demo/embeddings.json,Token embedding matrix export used by the dataset viewer and hub.


['agent.md',
 'checkpoints',
 'config.json',
 'corpus_sampling_log.json',
 'corpus_sampling_log_latest.json',
 'embeddings.json',
 'generated_train_runtime_summary_v7.json',
 'generated_train_runtime_v7.c',
 'ir1_train_forward.json',
 'ir1_train_report.json',
 'ir2_train_backward.json',
 'ir2_train_summary.json',
 'ir_report.html',
 'ir_train_invariants.json',
 'layout_train.json',
 'layout_train_audit.json',
 'libtrain.so',
 'memory_diagnostic_latest.json',
 'memory_headroom_train_latest.json',
 'operator_train_run.json',
 'profile_train_latest',
 'python_authoring_plan.json',
 'run_scope.json',
 'run_scope.md',
 'template_train.json',
 'train_e2e_latest.json',
 'train_exec_plan.json',
 'train_init_config.json',
 'training.md',
 'training_checkpoint_policy.json',
 'training_grad_norms.json',
 'training_loss_curve.json',
 'training_parity.json',
 'training_pipeline.json',
 'training_plan.json',
 'training_step_profile.json',
 'weights.bump',
 'weights_init.bump',
 'weights_manifest.jso